# pySIPNET — Getting Started Tutorial

This notebook walks through a complete end-to-end SIPNET run using pySIPNET:

1. Generate synthetic climate forcing
2. Configure model parameters for a temperate deciduous forest
3. Run the model
4. Inspect and plot the outputs
5. *(Optional)* Explore an interactive dashboard

> **Prerequisites**
>
> - pySIPNET installed: `pip install pysipnet`
> - SIPNET binary compiled: run `make sipnet-standard` from the repo root (see README)
> - *Dashboard section only*: `pip install pysipnet[viz]`

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pysipnet import ClimateDrivers, ModelPreset, SIPNETRunner
from pysipnet.parameters import (
    AllocationParams,
    InitialConditions,
    LeafPhysiologyParams,
    PhenologyParams,
    PhotosynthesisParams,
    RespirationParams,
    SIPNETParametersV1,
    WaterParams,
)

%matplotlib inline

## 1. Synthetic Climate

We generate two years of synthetic daily forcing for a temperate deciduous forest site.
All variables are computed from simple sinusoidal seasonal cycles plus noise — no
external data files required.

In [ ]:
rng = np.random.default_rng(42)

n_years = 2
start_year = 2020

doys = np.tile(np.arange(1, 366), n_years)
years = np.repeat(np.arange(start_year, start_year + n_years), 365)
theta = 2 * np.pi * (doys - 1) / 365  # seasonal phase angle

# Air and soil temperature (°C) — sinusoidal with noise
tair  = 7.5 + 12.5 * np.sin(theta - np.pi / 2) + rng.normal(0, 1.5, len(doys))
tsoil = 7.5 + 10.0 * np.sin(theta - np.pi / 2 - 0.3) + rng.normal(0, 0.5, len(doys))

# PAR (mol photons m⁻² per day) — zero in winter, peak in summer
par = np.maximum(0.0, 8 + 8 * np.sin(theta - np.pi / 2))
par *= np.maximum(0.5, 1 + rng.normal(0, 0.15, len(doys)))

# Precipitation (mm) — random events, 30 % occurrence
precip = rng.exponential(2.0, len(doys)) * (rng.random(len(doys)) < 0.3)

# VPD (Pa) — higher in summer
vpd = np.maximum(50.0, 500 + 600 * np.maximum(0, np.sin(theta - np.pi / 2))
                 + rng.normal(0, 100, len(doys)))
vpd_soil = np.maximum(50.0, vpd * 0.7 + rng.normal(0, 50, len(doys)))
vpress   = np.maximum(200.0, 1200 - vpd)

# Wind speed (m s⁻¹)
wspd = np.maximum(0.1, rng.exponential(2.0, len(doys)) + 0.5)

climate = ClimateDrivers.from_dataframe(pd.DataFrame({
    "year": years, "day": doys, "time": 0.0, "length": 1.0,
    "tair": tair, "tsoil": tsoil, "par": par, "precip": precip,
    "vpd": vpd, "vpd_soil": vpd_soil, "vpress": vpress, "wspd": wspd,
}))
climate

## 2. Model Parameters

`SIPNETParametersV1` groups parameters by domain (initial conditions,
photosynthesis, phenology, etc.).  We use the `STANDARD` preset which enables
snow tracking, growing-degree-day phenology, and moisture-sensitive
heterotrophic respiration.

In [ ]:
params = SIPNETParametersV1(
    initial_conditions=InitialConditions(
        plant_wood=30000.0,   # g C m⁻²  — aboveground + root wood
        lai=0.0,              # m² m⁻²   — starts bare, GDD drives leaf-out
        soil=10000.0,         # g C m⁻²
        soil_water_frac=0.5,  # fraction of WHC
        snow=1.0,             # cm water equiv.
        fine_root_frac=0.05,
        coarse_root_frac=0.15,
    ),
    photosynthesis=PhotosynthesisParams(
        a_max=112.0,           # nmol CO₂ g⁻¹ leaf s⁻¹
        a_max_frac=0.76,
        base_fol_resp_frac=0.1,
        psn_t_min=2.0,         # °C
        psn_t_opt=24.0,        # °C
        d_vpd_slope=0.05,
        d_vpd_exp=1.0,
        half_sat_par=300.0,    # mol m⁻² ground day⁻¹
        attenuation=0.5,
    ),
    phenology=PhenologyParams(
        gdd_leaf_on=100.0,     # °C·day accumulated from Jan 1
        leaf_off_day=270.0,    # DOY
        leaf_growth=50.0,      # g C m⁻²
        frac_leaf_fall=0.95,
        leaf_allocation=0.25,
        leaf_turnover_rate=1.0,  # year⁻¹
    ),
    respiration=RespirationParams(
        base_veg_resp=0.02,
        veg_resp_q10=2.0,
        growth_resp_frac=0.0,
        frozen_soil_fol_r_eff=0.5,
        frozen_soil_threshold=-1.0,
        base_fine_root_resp=0.5,
        base_coarse_root_resp=0.1,
        fine_root_q10=2.0,
        coarse_root_q10=2.0,
        base_soil_resp=0.06,
        soil_resp_q10=2.0,
        soil_resp_moist_effect=1.5,
    ),
    allocation=AllocationParams(
        fine_root_allocation=0.35,
        wood_allocation=0.30,
        fine_root_turnover_rate=1.0,    # year⁻¹
        coarse_root_turnover_rate=0.1,  # year⁻¹
        wood_turnover_rate=0.02,        # year⁻¹
    ),
    water=WaterParams(
        water_remove_frac=0.1,
        frozen_soil_eff=0.1,
        wue_const=10.0,
        soil_whc=12.0,   # cm
        litter_whc=5.0,  # cm
        immed_evap_frac=0.1,
        fast_flow_frac=0.1,
        snow_melt=0.15,  # cm °C⁻¹ day⁻¹
        rd_const=100.0,
        r_soil_const1=3.0,
        r_soil_const2=2.0,
    ),
    leaf=LeafPhysiologyParams(
        leaf_c_sp_wt=32.0,  # g C m⁻² leaf
        c_frac_leaf=0.45,
    ),
)

## 3. Run the Model

`SIPNETRunner.run()` writes inputs to a fresh temporary directory, executes the
SIPNET binary, and returns a `SIPNETResult` containing the parsed output.

In [ ]:
runner = SIPNETRunner(preset=ModelPreset.STANDARD)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # suppress low-VPD warnings from synthetic data
    result = runner.run(params, climate)

print(f"Success   : {result.success}")
print(f"Timesteps : {len(result.timeseries)}")

## 4. Inspect Outputs

The `timeseries` attribute is a pandas DataFrame with one row per model
timestep and snake_case column names.

In [ ]:
result.timeseries.head()

In [ ]:
result.timeseries[["nee", "gpp", "evapotranspiration", "soil_c", "plant_wood_c"]].describe()

## 5. Visualize Outputs

In [ ]:
ts = result.timeseries
x = ts["year"] + (ts["day"] - 1) / 365

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

# Carbon fluxes
axes[0].plot(x, ts["nee"], label="NEE", color="steelblue", lw=0.9)
axes[0].plot(x, ts["gpp"], label="GPP", color="forestgreen", lw=0.9)
axes[0].axhline(0, color="k", lw=0.5, ls="--")
axes[0].set_ylabel("g C m⁻² per day")
axes[0].set_title("Carbon Fluxes")
axes[0].legend(frameon=False)

# Evapotranspiration
axes[1].plot(x, ts["evapotranspiration"], color="cornflowerblue", lw=0.9)
axes[1].set_ylabel("cm per day")
axes[1].set_title("Evapotranspiration")

# Carbon pools
axes[2].plot(x, ts["soil_c"],       label="Soil C",       color="sienna",    lw=0.9)
axes[2].plot(x, ts["plant_wood_c"], label="Plant Wood C", color="darkgreen", lw=0.9)
axes[2].set_ylabel("g C m⁻²")
axes[2].set_xlabel("Year")
axes[2].set_title("Carbon Pools")
axes[2].legend(frameon=False)

fig.tight_layout()
plt.show()

---

## 6. Interactive Dashboard *(optional — requires `pip install pysipnet[viz]`)*

`pysipnet.viz.dashboard` builds a Plotly figure summarising climate inputs,
fluxes, and pools in a single interactive view.  Users who have not installed
the `viz` extra can skip this section.

In [ ]:
from pysipnet.viz import dashboard

fig = dashboard(result)
fig.show()